In [ ]:
!pip install transformers pandas torch scikit-learn
!pip install nltk spacy word2number
!python -m spacy download en_core_web_sm
!pip install -U transformers

In [ ]:
# --- Imports ---
import os
import random
import re
import string
from collections import Counter
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer
import nltk
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from transformers import (BertTokenizer, RobertaTokenizer, Trainer, TrainingArguments,
                          EarlyStoppingCallback, BertModel, RobertaModel, TrainerCallback)
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Setup ---
os.environ["WANDB_DISABLED"] = "true"
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# --- Text Processing ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [word for word in text.split() if word not in stop_words]
    text = ' '.join(stemmer.stem(word) for word in tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != word:
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, n=1):
    words = text.split()
    new_words = words.copy()
    random_word_list = list(set([word for word in words if word not in stop_words and len(word) > 3]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)

def phrase_insertion(text, n=1):
    phrases = ["the system shall", "must be able to", "should ensure that"]
    words = text.split()
    for _ in range(n):
        add_phrase = random.choice(phrases)
        random_idx = random.randint(0, len(words)-1)
        words.insert(random_idx, add_phrase)
    return ' '.join(words)

def text_simplification(text):
    text = re.sub(r'\bthe system shall\b', '', text)
    text = re.sub(r'\bmust be able to\b', '', text)
    text = re.sub(r'\bshould ensure that\b', '', text)
    return text.strip()

def random_deletion(text, p=0.1):
    words = text.split()
    if len(words) == 1:
        return text
    remaining = [word for word in words if random.random() > p]
    return ' '.join(remaining) if remaining else random.choice(words)

def random_swap(text, n=1):
    words = text.split()
    if len(words) < 2:
        return text
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return ' '.join(words)

# --- Data Loading: PROMISE expanded split ---
DATA_PATH_CANDIDATES = [
    os.path.join('..', 'data', 'exp', 'promise_exp.csv'),
    os.path.join('data', 'exp', 'promise_exp.csv'),
    r'D:\semester4\SWR302\RBL_requirements_classification\rbl-requirements-classification\data\exp\promise_exp.csv',
]

for data_path in DATA_PATH_CANDIDATES:
    if os.path.exists(data_path):
        break
else:
    raise FileNotFoundError(
        "Could not find promise_exp.csv. Expected it under data/exp/promise_exp.csv."
    )

df = pd.read_csv(data_path)
df = df.rename(columns={'RequirementText': 'text', 'class': 'label'})
df = df[['text', 'label']].dropna().reset_index(drop=True)
df = df[df['label'].map(df['label'].value_counts()) > 1].reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['label'],
    random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Loaded PROMISE expanded dataset from: {data_path}")
print("Train Distribution:", dict(Counter(train_df['label'])))
print("Validation Distribution:", dict(Counter(val_df['label'])))
print("Test Distribution:", dict(Counter(test_df['label'])))

train_df['cleaned_text'] = train_df['text'].apply(lambda x: clean_text(str(x)))
val_df['cleaned_text'] = val_df['text'].apply(lambda x: clean_text(str(x)))
test_df['cleaned_text'] = test_df['text'].apply(lambda x: clean_text(str(x)))

# --- Augmentation: train split only, exact target without overshoot ---
augmentation_targets = {
    'SE': 200, 'O': 200, 'PE': 200, 'US': 200, 'LF': 200,
    'A': 200, 'MN': 200, 'L': 200, 'SC': 200, 'FT': 200,
    'PO': 200, 'F': 200
}
augmented = []

for cls, min_required in augmentation_targets.items():
    class_df = train_df[train_df['label'] == cls]
    needed = max(0, min_required - len(class_df))
    if len(class_df) == 0 or needed == 0:
        continue
    for aug_idx in range(needed):
        row = class_df.sample(n=1, replace=True, random_state=RANDOM_STATE + aug_idx).iloc[0]
        text = row['cleaned_text']
        aug_text = synonym_replacement(text, 1)
        aug_text = phrase_insertion(aug_text, 1)
        aug_text = text_simplification(aug_text)
        aug_text = random_swap(aug_text, 1)
        aug_text = random_deletion(aug_text, p=0.1)
        augmented.append({
            'text': aug_text,
            'cleaned_text': aug_text,
            'label': cls,
            'source': 'AUG'
        })

df_aug = pd.DataFrame(augmented)
df_combined = pd.concat([train_df, df_aug], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("Combined Train Distribution:", dict(Counter(df_combined['label'])))

# --- Extract Texts & Labels ---
train_texts = df_combined['cleaned_text'].tolist()
train_labels_raw = df_combined['label'].tolist()
val_texts = val_df['cleaned_text'].tolist()
val_labels_raw = val_df['label'].tolist()
test_texts = test_df['cleaned_text'].tolist()
test_labels_raw = test_df['label'].tolist()

# --- Label Encoding ---
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_labels_raw)
val_labels = label_encoder.transform(val_labels_raw)
test_labels = label_encoder.transform(test_labels_raw)

# --- Tokenization ---
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize(tokenizer, texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=128)

bert_train_enc = tokenize(bert_tokenizer, train_texts)
roberta_train_enc = tokenize(roberta_tokenizer, train_texts)
bert_val_enc = tokenize(bert_tokenizer, val_texts)
roberta_val_enc = tokenize(roberta_tokenizer, val_texts)
bert_test_enc = tokenize(bert_tokenizer, test_texts)
roberta_test_enc = tokenize(roberta_tokenizer, test_texts)

# --- Dataset ---
class HybridDataset(Dataset):
    def __init__(self, bert_enc, roberta_enc, labels):
        self.bert = bert_enc
        self.roberta = roberta_enc
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'bert_input_ids': torch.tensor(self.bert['input_ids'][idx]),
            'bert_attention_mask': torch.tensor(self.bert['attention_mask'][idx]),
            'roberta_input_ids': torch.tensor(self.roberta['input_ids'][idx]),
            'roberta_attention_mask': torch.tensor(self.roberta['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

train_dataset = HybridDataset(bert_train_enc, roberta_train_enc, train_labels)
val_dataset = HybridDataset(bert_val_enc, roberta_val_enc, val_labels)
test_dataset = HybridDataset(bert_test_enc, roberta_test_enc, test_labels)

# Model
class HybridBertRobertaClassifier(torch.nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = torch.nn.Dropout(0.5)
        self.bert_reduce = torch.nn.Linear(768, 256)
        self.roberta_reduce = torch.nn.Linear(768, 256)
        self.batchnorm = torch.nn.BatchNorm1d(512)
        self.classifier = torch.nn.Linear(512, num_labels, bias=False)

        for name, param in self.bert.named_parameters():
            if name.startswith("encoder.layer.") and int(name.split(".")[2]) < 8:
                param.requires_grad = False
        for name, param in self.roberta.named_parameters():
            if name.startswith("encoder.layer.") and int(name.split(".")[2]) < 8:
                param.requires_grad = False

    def forward(self, bert_input_ids, bert_attention_mask, roberta_input_ids, roberta_attention_mask, labels=None):
        bert_output = self.bert(input_ids=bert_input_ids, attention_mask=bert_attention_mask).pooler_output
        roberta_output = self.roberta(input_ids=roberta_input_ids, attention_mask=roberta_attention_mask).pooler_output
        x = torch.cat([self.bert_reduce(self.dropout(bert_output)), self.roberta_reduce(self.dropout(roberta_output))], dim=1)
        x = self.batchnorm(x)
        logits = self.classifier(x)
        loss = torch.nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits}

# Collator
def hybrid_collator(batch):
    return {
        'bert_input_ids': torch.stack([x['bert_input_ids'] for x in batch]),
        'bert_attention_mask': torch.stack([x['bert_attention_mask'] for x in batch]),
        'roberta_input_ids': torch.stack([x['roberta_input_ids'] for x in batch]),
        'roberta_attention_mask': torch.stack([x['roberta_attention_mask'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch])
    }

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': p,
        'recall': r,
        'f1': f1,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
    }

# Accuracy tracking
class AccuracyPlotCallback(TrainerCallback):
    def __init__(self):
        self.train_accuracies = []
        self.val_accuracies = []
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, args, state, control, **kwargs):
        train_metrics = trainer.evaluate(train_dataset)
        val_metrics = trainer.evaluate(val_dataset)
        self.train_accuracies.append(train_metrics["eval_accuracy"])
        self.val_accuracies.append(val_metrics["eval_accuracy"])
        self.train_losses.append(train_metrics["eval_loss"])
        self.val_losses.append(val_metrics["eval_loss"])

accuracy_callback = AccuracyPlotCallback()

# Training
training_args = TrainingArguments(
    output_dir='./results/hybrid_bert_roberta',
    num_train_epochs=30,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.02,
    learning_rate=1e-5,
    logging_dir='./logs/hybrid_bert_roberta',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    gradient_accumulation_steps=2,
    fp16=True,
    dataloader_num_workers=4,
    report_to="none",
    max_grad_norm=1.0,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = HybridBertRobertaClassifier(num_labels=len(label_encoder.classes_)).to(device)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=hybrid_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2), accuracy_callback]
)

trainer.train()

# Evaluation
print("Validation Results:")
eval_results = trainer.evaluate(val_dataset)
print(eval_results)

print("\nTesting Results:")
test_results = trainer.predict(test_dataset)
test_preds = test_results.predictions.argmax(-1)
print(classification_report(test_labels, test_preds, target_names=label_encoder.classes_, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Counts)")
plt.show()

cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
plt.figure(figsize=(10, 7))
sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix (%)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Accuracy/Loss plots
epochs = range(1, len(accuracy_callback.train_accuracies)+1)
plt.figure(figsize=(10, 6))
plt.plot(epochs, accuracy_callback.train_accuracies, 'r-o', label='Train Accuracy')
plt.plot(epochs, accuracy_callback.val_accuracies, 'b-o', label='Val Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Val Accuracy")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(epochs, accuracy_callback.train_losses, 'r-o', label='Train Loss')
plt.plot(epochs, accuracy_callback.val_losses, 'b-o', label='Val Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.grid()
plt.show()

